# Station selection

The project forecasts daily soil moisture (`cosmos_vwc`) at **one** COSMOS-UK station.
This notebook ranks all stations by data quality over the backfill window and records why
the project uses **BALRD** (Balruddery, Scotland).

What a good station needs:

- **Complete target.** `cosmos_vwc` present nearly every day, with no long outages
  (a long gap breaks the lag features for weeks).
- **Complete covariates.** `precip`, `ta`, `rh`, `stp_tsoil20` rarely missing.
- **Plausible values.** Few readings outside physical ranges.
- **Something to learn.** Enough variation in soil moisture; the lag-2 persistence error
  shows how hard the naive baseline is to beat.

This is the notebook version of `src/scripts/get_station.py`; the saved result is
`src/scripts/station_ranking.csv`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import time
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

## Settings

Daily collection (`1D`), the five parameters the pipeline uses, and the plausible ranges
(the same ones as in `config/settings.toml`). The window is 2023-01-01 to 2026-10-29,
the day the ranking was made.

In [ ]:
BASE = "https://cosmos-api.ceh.ac.uk"
COLLECTION = "1D"  # daily data
PARAMS = ["cosmos_vwc", "precip", "ta", "rh", "stp_tsoil20"]
TARGET = "cosmos_vwc"
FEATURE_COLS = ["precip", "ta", "rh", "stp_tsoil20"]
RANGES = {  # plausible physical ranges, used to flag odd values
    "cosmos_vwc": (0, 100),
    "precip": (0, 300),
    "ta": (-30, 45),
    "rh": (0, 100),
    "stp_tsoil20": (-10, 40),
}
START = date(2023, 1, 1)
END = date(2026, 10, 29)

# Recompute the ranking from the API (slow: one request per station and year)
# or load the saved CSV.
RECOMPUTE = False
RANKING_CSV = Path("../src/scripts/station_ranking.csv")

CACHE = Path("../src/scripts/cosmos_cache")  # gitignored
CACHE.mkdir(exist_ok=True)
HEADERS = {"Accept": "application/json, application/prs.coverage+json"}

## HTTP helper

Every response is cached on disk, so rerunning the notebook does not hit the API again.
A 404 means the station has no data for that window.

In [ ]:
def get_json(url: str, params: dict | None = None, retries: int = 4, timeout: int = 60):
    key = hashlib.md5((url + json.dumps(params or {}, sort_keys=True)).encode()).hexdigest()
    cache_file = CACHE / f"{key}.json"
    if cache_file.exists():
        return json.loads(cache_file.read_text())
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=timeout)
            if r.status_code == 404:
                return None  # station has no data for this window
            r.raise_for_status()
            js = r.json()
            cache_file.write_text(json.dumps(js))
            return js
        except (requests.RequestException, ValueError) as e:
            last = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed after {retries} tries: {url} ({last})")

## Station list

`GET /collections/1D/locations` returns GeoJSON, one feature per station.

In [ ]:
def list_stations() -> list[dict]:
    js = get_json(f"{BASE}/collections/{COLLECTION}/locations")
    feats = js.get("features", []) if isinstance(js, dict) else []
    out = []
    for f in feats:
        props = f.get("properties", {}) or {}
        sid = f.get("id") or props.get("id") or props.get("identifier") or props.get("name")
        coords = (f.get("geometry") or {}).get("coordinates")
        out.append({"id": str(sid), "coords": coords, "props": props, "text": json.dumps(f).lower()})
    return out


stations = list_stations()
print(f"{len(stations)} stations found. Metadata of the first one:")
print(json.dumps(stations[0]["props"], indent=2)[:1500])

## Look at one raw response first

Before scoring, check the format of a data response: CoverageJSON with a time axis `t`
and one range per parameter. Ten days of BALRD are enough.

In [ ]:
def coverage_to_df(js) -> pd.DataFrame:
    """Convert a CoverageJSON (or collection of coverages) to a daily DataFrame."""
    if js is None:
        return pd.DataFrame(columns=PARAMS)
    covs = js.get("coverages") if js.get("type") == "CoverageCollection" else [js]
    frames = []
    for cov in covs:
        axes = cov["domain"]["axes"]
        times = pd.to_datetime(axes["t"]["values"], utc=True)
        data = {}
        for name, rng in cov["ranges"].items():
            vals = rng["values"]
            if len(vals) != len(times):  # non-time-aligned range, skip
                continue
            data[name] = pd.to_numeric(pd.Series(vals, dtype="object"), errors="coerce").to_numpy()
        frames.append(pd.DataFrame(data, index=times))
    if not frames:
        return pd.DataFrame(columns=PARAMS)
    df = pd.concat(frames)
    df.index = df.index.tz_convert("UTC").normalize().tz_localize(None)
    df.index.name = "date"
    return df


js = get_json(
    f"{BASE}/collections/{COLLECTION}/locations/BALRD",
    params={"datetime": "2023-01-01T00:00:00Z/2023-01-11T00:00:00Z", "parameter-name": ",".join(PARAMS)},
)
print(json.dumps(js, indent=2)[:2500])
coverage_to_df(js).head()

## Fetch a station

One request per calendar year, the same way the feature pipeline backfills.

In [ ]:
def fetch_station(sid: str, start: date, end: date) -> pd.DataFrame:
    frames = []
    for year in range(start.year, end.year + 1):
        y0 = max(start, date(year, 1, 1))
        y1 = min(end, date(year, 12, 31))
        dt = f"{y0.isoformat()}T00:00:00Z/{y1.isoformat()}T00:00:00Z"
        js = get_json(
            f"{BASE}/collections/{COLLECTION}/locations/{sid}",
            params={"datetime": dt, "parameter-name": ",".join(PARAMS)},
        )
        frames.append(coverage_to_df(js))
    df = pd.concat(frames) if frames else pd.DataFrame(columns=PARAMS)
    df = df[~df.index.duplicated(keep="last")].sort_index()
    for c in PARAMS:
        if c not in df:
            df[c] = np.nan
    return df

## Scoring

Per station, over the full daily index of the window:

| Column | Meaning |
|---|---|
| `vwc_coverage` | share of days with a `cosmos_vwc` value |
| `n_gaps`, `longest_gap_days` | runs of missing target days |
| `<param>_missing` | share of days missing for each covariate |
| `out_of_range_values` | readings outside `RANGES`, all parameters |
| `vwc_std` | variation of the target |
| `persistence_mae_lag2` | MAE of predicting obs(d) with obs(d-2), the project's naive baseline |

The score is coverage minus penalties:

- longest gap / 60, capped at 0.3 (a long outage hurts lag features),
- half the mean covariate missing share,
- out-of-range share, capped at 0.2.

In [ ]:
def gap_stats(missing: pd.Series) -> tuple[int, int]:
    """(number of gaps, longest gap in days) from a boolean 'missing' series."""
    if not missing.any():
        return 0, 0
    grp = (missing != missing.shift()).cumsum()
    runs = missing.groupby(grp).agg(["first", "size"])
    runs = runs[runs["first"]]
    return len(runs), int(runs["size"].max())


def score_station(sid: str, df: pd.DataFrame, start: date, end: date) -> dict:
    full_idx = pd.date_range(start, end, freq="D")
    df = df.reindex(full_idx)
    n_days = len(full_idx)

    y = df[TARGET]
    miss = y.isna()
    n_gaps, longest = gap_stats(miss)
    first_valid = y.first_valid_index()

    feat_missing = {c: float(df[c].isna().mean()) for c in FEATURE_COLS}
    out_of_range = 0
    for c, (lo, hi) in RANGES.items():
        out_of_range += int(((df[c] < lo) | (df[c] > hi)).sum())

    # persistence baseline for the project horizon: predict obs(d) with obs(d-2)
    pers = (y - y.shift(2)).abs().dropna()
    pers_mae = float(pers.mean()) if len(pers) else np.nan

    coverage = 1 - miss.mean()
    score = (
        coverage
        - min(longest / 60, 0.3)                      # long outage hurts lag features
        - 0.5 * float(np.mean(list(feat_missing.values())))
        - min(out_of_range / n_days, 0.2)
    )
    return {
        "station": sid,
        "days_in_window": n_days,
        "first_vwc_date": first_valid.date() if first_valid is not None else None,
        "vwc_coverage": round(coverage, 4),
        "vwc_missing_days": int(miss.sum()),
        "n_gaps": n_gaps,
        "longest_gap_days": longest,
        **{f"{c}_missing": round(v, 4) for c, v in feat_missing.items()},
        "out_of_range_values": out_of_range,
        "vwc_std": round(float(y.std()), 3) if y.notna().any() else np.nan,
        "persistence_mae_lag2": round(pers_mae, 3) if pers_mae == pers_mae else np.nan,
        "score": round(score, 4),
    }

## Rank all stations

With `RECOMPUTE = False` this loads the saved ranking. Set it to `True` to rebuild it
from the API (cached responses are reused).

In [ ]:
if RECOMPUTE:
    rows = []
    for i, s in enumerate(stations, 1):
        sid = s["id"]
        try:
            rows.append(score_station(sid, fetch_station(sid, START, END), START, END))
            r = rows[-1]
            print(f"[{i}/{len(stations)}] {sid}: coverage={r['vwc_coverage']} longest_gap={r['longest_gap_days']}")
        except Exception as e:  # keep going if one station fails
            print(f"[{i}/{len(stations)}] {sid}: FAILED ({e})")
    ranking = pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)
    ranking.to_csv(RANKING_CSV, index=False)
else:
    ranking = pd.read_csv(RANKING_CSV)

print(f"{len(ranking)} of {len(stations)} stations scored")
ranking.head(10)

## Overview

How many stations clear the bar, and how many are unusable.

In [ ]:
summary = pd.Series({
    "stations scored": len(ranking),
    "100% vwc coverage, no gaps": int(((ranking["vwc_coverage"] == 1) & (ranking["n_gaps"] == 0)).sum()),
    "longest gap > 30 days": int((ranking["longest_gap_days"] > 30).sum()),
    "no vwc data at all": int((ranking["vwc_coverage"] == 0).sum()),
})
summary

Twenty stations have a complete target over the whole window, so coverage alone does not
decide. Among those, the score separates them by covariate gaps and out-of-range values.
Stations at the bottom (e.g. CARDT, WRTTL, CRICH, and four with no data) are out.

## Top candidates

The top three by score, compared side by side.

In [ ]:
top = ranking.head(3)["station"].tolist()
ranking.set_index("station").loc[top].T

A look at the raw series for the top three: are there flat lines, spikes or step changes
the scores would miss?

In [ ]:
series = {sid: fetch_station(sid, START, END)[TARGET] for sid in top}

fig, axes = plt.subplots(len(top), 1, figsize=(10, 2.4 * len(top)), sharex=True, sharey=True)
for ax, sid in zip(axes, top):
    ax.plot(series[sid].index, series[sid].values, color="#2a78d6", linewidth=1.5)
    ax.set_title(sid, loc="left", fontsize=11)
    ax.set_ylabel("vwc (%)")
    ax.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
fig.suptitle("Daily soil moisture (cosmos_vwc), 2023-01-01 to 2026-10-29", x=0.01, ha="left")
fig.tight_layout()
plt.show()

## BALRD metadata

In [ ]:
balrd = next(s for s in stations if s["id"].upper() == "BALRD")
print("coords (lat, lon as returned by the API):", balrd["coords"])
print(json.dumps(balrd["props"], indent=2))

## Decision: BALRD

BALRD (Balruddery, Scotland; lat 56.482297, lon -3.1114881, 130 m) ranks second with a
score of 0.9997, effectively tied with NWYKE (0.9998). Over 2023-01-01 to 2026-10-29:

- 1,398 daily records, 100 % `cosmos_vwc` coverage, no gaps,
- 0.2 % missing `precip`, no missing `ta`, `rh` or `stp_tsoil20`,
- no out-of-range values,
- record starts 2014-05-15, so the backfill can be extended later.

The station is a config value (`station_id` in `config/settings.toml`), so switching
station later only needs a new entry there.